In [4]:
import requests
from typing import Dict, Tuple, Any, Optional

CKAN_URL = "https://catalogodatos.cnmc.es"
HEADERS = {"User-Agent": "Application"}

MIN_YEAR = 2016
MAX_YEAR = 2025

# =========================
# CKAN helpers
# =========================
def _package_search(query: str, rows: int = 20) -> Dict[str, Any]:
    """Llama a CKAN package_search y devuelve el JSON (parseado)."""
    url = f"{CKAN_URL}/api/3/action/package_search"
    params = {"q": f'"{query}"', "rows": rows}  # comillas => frase exacta
    r = requests.get(url, headers=HEADERS, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def _datastore_search_smoke_test(resource_id: str) -> bool:
    """
    Prueba rápida: verifica que el resource_id funciona con datastore_search (limit=1).
    Devuelve True si responde success=True, False si falla (HTTP/CKAN/estructura).
    """
    url = f"{CKAN_URL}/api/3/action/datastore_search"
    params = {"resource_id": resource_id, "limit": 1, "offset": 0}
    try:
        r = requests.get(url, headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        j = r.json()
        return bool(j.get("success", False))
    except Exception:
        return False

# =========================
# Robust picker
# =========================
def _pick_resource_id_robusto(results: list, expected_title: Optional[str] = None) -> Tuple[str, bool]:
    """
    Elige un resource_id de forma robusta.

    Estrategia:
    1) Intentar seleccionar dataset por title (exacto y luego "contiene", case-insensitive)
    2) Si no, usar results[0] (fallback)
    3) Preferir resource con datastore_active=True
    4) Si no hay datastore_active, caer a resources[0].id (fallback web) y marcarlo como fallback=True

    Returns:
    (resource_id, used_fallback)
    """
    if not results:
        raise ValueError("No hubo resultados en package_search (results vacío).")

    dataset = None

    if expected_title:
        # 1) Match exacto por title
        for ds in results:
            if ds.get("title") == expected_title:
                dataset = ds
                break

        # 2) Match por contiene (tolerante)
        if dataset is None:
            expected_lower = expected_title.lower()
            for ds in results:
                title = (ds.get("title") or "").lower()
                if expected_lower in title:
                    dataset = ds
                    break

    # 3) Fallback dataset: el primero
    if dataset is None:
        dataset = results[0]

    resources = dataset.get("resources") or []
    if not resources:
        raise ValueError("Dataset encontrado, pero no tiene resources.")

    # Preferir un resource que esté en DataStore
    for res in resources:
        if res.get("datastore_active") is True and res.get("id"):
            return res["id"], False

    # Fallback al primer recurso, como indica la web
    first_id = resources[0].get("id")
    if first_id:
        return first_id, True

    raise ValueError("No se pudo determinar un resource_id válido (resources[0] sin id).")

# =========================
# Main function
# =========================
def build_resource_map(start_year: int, end_year: int) -> Tuple[Dict[int, str], Dict[int, str]]:
    """
    Construye un diccionario {año: resource_id} para:
    'Precios diarios provinciales - {year} -'
    entre start_year y end_year (incluidos), validando rango [2016, 2025].

    Robustez:
    - Selecciona dataset por title si se puede (expected_title)
    - Prefiere datastore_active=True
    - Si cae a fallback resources[0].id, valida con datastore_search(limit=1):
        - si falla, registra error y salta ese año
    - Si un año falla, lo registra en errores y continúa.

    Devuelve: (ids_por_anio, errores_por_anio)
    """
    # Validaciones del rango
    if not (MIN_YEAR <= start_year <= MAX_YEAR):
        raise ValueError(f"start_year debe estar entre {MIN_YEAR} y {MAX_YEAR}. Recibido: {start_year}")
    if not (MIN_YEAR <= end_year <= MAX_YEAR):
        raise ValueError(f"end_year debe estar entre {MIN_YEAR} y {MAX_YEAR}. Recibido: {end_year}")
    if start_year > end_year:
        raise ValueError(f"start_year ({start_year}) no puede ser mayor que end_year ({end_year}).")

    ids_por_anio: Dict[int, str] = {}
    errores_por_anio: Dict[int, str] = {}

    for year in range(start_year, end_year + 1):
        expected = f"Precios diarios provinciales - {year} -"

        try:
            datos_catalogo = _package_search(query=expected, rows=20)

            if not datos_catalogo.get("success", False):
                errores_por_anio[year] = "CKAN devolvió success=False."
                print(f"[{year}] ERROR: CKAN devolvió success=False")
                continue

            result = datos_catalogo.get("result", {}) or {}
            results = result.get("results", []) or []
            count = result.get("count", 0)

            if count == 0 or not results:
                errores_por_anio[year] = f"No se encontraron datasets con query: {expected}"
                print(f"[{year}] ERROR: No se encontraron resultados (count=0).")
                continue

            resource_id, used_fallback = _pick_resource_id_robusto(results, expected_title=expected)

            # Si usamos fallback (resources[0].id), validamos que sea realmente DataStore (datastore_search funcione)
            if used_fallback:
                ok = _datastore_search_smoke_test(resource_id)
                if not ok:
                    errores_por_anio[year] = (
                        "Se eligió resources[0].id como fallback, pero datastore_search(limit=1) falló. "
                        f"resource_id={resource_id}"
                    )
                    print(f"[{year}] ERROR: fallback resource_id NO funciona con datastore_search -> {resource_id}")
                    continue
                print(f"[{year}] OK (fallback validado) -> resource_id: {resource_id}")
            else:
                print(f"[{year}] OK -> resource_id: {resource_id}")

            ids_por_anio[year] = resource_id

        except requests.exceptions.RequestException as e:
            errores_por_anio[year] = f"Error HTTP/red: {repr(e)}"
            print(f"[{year}] ERROR HTTP/red: {e}")
            continue
        except Exception as e:
            errores_por_anio[year] = f"Error: {repr(e)}"
            print(f"[{year}] ERROR: {e}")
            continue

    print("\n=== Resumen ===")
    print(f"Años OK: {len(ids_por_anio)} -> {sorted(ids_por_anio.keys())}")
    print(f"Años con error: {len(errores_por_anio)} -> {sorted(errores_por_anio.keys())}")

    return ids_por_anio, errores_por_anio

In [5]:
ids_por_anio, errores_por_anio = build_resource_map(2016, 2025)
ids_por_anio
errores_por_anio

[2016] OK -> resource_id: a385ec5d-a22b-4029-a322-ce3f40241597
[2017] OK -> resource_id: 4c94e9aa-4973-471c-ae19-6658ec57e865
[2018] OK -> resource_id: e2a074ed-789e-43fc-b7bf-e2ba6106458a
[2019] OK -> resource_id: 898c5d4b-c78b-4653-9226-bc24de59846a
[2020] OK -> resource_id: beb221c5-2be6-472a-bb25-bd6d2343e014
[2021] OK -> resource_id: 9bb7d9fe-b99a-42ea-96f7-35c735b56612
[2022] OK -> resource_id: 42fca586-6582-40c8-8df5-6ebbb8fbfd73
[2023] OK -> resource_id: b5a89db0-239f-4c8a-bd98-6575858359ae
[2024] OK -> resource_id: 141fdb3b-7c56-4eed-bf8d-bee56e577aa6
[2025] OK -> resource_id: 510d138a-6c6d-4dce-8d30-8c77de58d787

=== Resumen ===
Años OK: 10 -> [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Años con error: 0 -> []


{}

In [ ]:
import requests
import pandas as pd
import time
import os
from typing import Dict

CKAN_URL = "https://catalogodatos.cnmc.es"
HEADERS = {"User-Agent": "IA-Proyecto-OPEC"}

# Carpeta donde guardaremos los CSV
os.makedirs("../data", exist_ok=True)

def descargar_recurso_completo(resource_id: str, limit: int = 5000) -> list:
    """
    Descarga TODOS los registros de un resource CKAN usando paginación (limit/offset).
    """
    registros = []
    offset = 0

    while True:
        url = f"{CKAN_URL}/api/3/action/datastore_search"
        params = {"resource_id": resource_id, "limit": limit, "offset": offset}

        r = requests.get(url, headers=HEADERS, params=params, timeout=60)
        r.raise_for_status()

        data = r.json()["result"]["records"]
        if not data:
            break

        registros.extend(data)
        offset += limit
        print(f"⬇ Descargados {len(registros)} registros hasta ahora...")
        time.sleep(0.2)

    return registros

def descargar_limpiar_y_guardar(resource_id: str, year: int) -> None:
    """
    Descarga el dataset (por resource_id), lo convierte a DataFrame, lo limpia y lo guarda
    como ../data/precios_petroleo_{year}_limpio.csv
    """
    # Descargar todos los datos del recurso
    datos = descargar_recurso_completo(resource_id)

    # Convertir a DataFrame
    df = pd.DataFrame(datos)

    # Limpiar espacios de los nombres de columnas
    df.columns = df.columns.str.strip()

    # Renombrar columnas (como lo tenías)
    df.rename(columns={
        "fecha_precio": "fecha",
        "provincia": "provincia",
        "producto": "producto",
        "promedio_de_pai_diario_cubo": "pai",
        "promedio_de_pvp_diario_cubo": "pvp"
    }, inplace=True)

    # Convertir tipos correctamente
    df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d", errors="coerce")
    df["pai"] = pd.to_numeric(df["pai"], errors="coerce")
    df["pvp"] = pd.to_numeric(df["pvp"], errors="coerce")

    # Eliminar filas con nulos
    df.dropna(inplace=True)

    # Guardar CSV final limpio
    out_path = f"../data/precios_petroleo_{year}_limpio.csv"
    df.to_csv(out_path, index=False)

    print(f"Datos guardados en '{out_path}' ({len(df)} registros)")

def descargar_por_anios(ids_por_anio: Dict[int, str]) -> None:
    """
    Recibe un diccionario {year: resource_id} y descarga/guarda cada año en orden.
    """
    for year in sorted(ids_por_anio.keys()):
        resource_id = ids_por_anio[year]
        print(f"\n===== Procesando año {year} =====")
        descargar_limpiar_y_guardar(resource_id, year)

    print("\nProceso terminado: todos los años fueron procesados.")





===== Procesando año 2016 =====
⬇ Descargados 5000 registros hasta ahora...
⬇ Descargados 10000 registros hasta ahora...
⬇ Descargados 15000 registros hasta ahora...
⬇ Descargados 20000 registros hasta ahora...
⬇ Descargados 25000 registros hasta ahora...
⬇ Descargados 30000 registros hasta ahora...
⬇ Descargados 35000 registros hasta ahora...
⬇ Descargados 40000 registros hasta ahora...
⬇ Descargados 45000 registros hasta ahora...
⬇ Descargados 50000 registros hasta ahora...
⬇ Descargados 55000 registros hasta ahora...
⬇ Descargados 60000 registros hasta ahora...
⬇ Descargados 65000 registros hasta ahora...
⬇ Descargados 70000 registros hasta ahora...
⬇ Descargados 75000 registros hasta ahora...
⬇ Descargados 75762 registros hasta ahora...
✅ Datos guardados en '../data/precios_petroleo_2016_limpio.csv' (75762 registros)

===== Procesando año 2017 =====
⬇ Descargados 5000 registros hasta ahora...
⬇ Descargados 10000 registros hasta ahora...
⬇ Descargados 15000 registros hasta ahora...

In [ ]:
# === USO ===
# cuando ya se tienen los IDs ids_por_anio del paso anterior (build_resource_map)
descargar_por_anios(ids_por_anio)

# Si  se quiere probar con un año puntual deberiamos buscarlo en el build_resource_map que se ejecuto antes
# Por ejemplo, para 2020:
#descargar_limpiar_y_guardar(ids_por_anio[2020], 2020)


⬇ Descargados 5000 registros hasta ahora...
⬇ Descargados 10000 registros hasta ahora...
⬇ Descargados 15000 registros hasta ahora...
⬇ Descargados 20000 registros hasta ahora...
⬇ Descargados 25000 registros hasta ahora...
⬇ Descargados 30000 registros hasta ahora...
⬇ Descargados 35000 registros hasta ahora...
⬇ Descargados 40000 registros hasta ahora...
⬇ Descargados 45000 registros hasta ahora...
⬇ Descargados 50000 registros hasta ahora...
⬇ Descargados 55000 registros hasta ahora...
⬇ Descargados 60000 registros hasta ahora...
⬇ Descargados 65000 registros hasta ahora...
⬇ Descargados 70000 registros hasta ahora...
⬇ Descargados 75000 registros hasta ahora...
⬇ Descargados 75762 registros hasta ahora...
✅ Datos guardados en '../data/precios_petroleo_2020_limpio.csv' (75762 registros)
